### Библиотеки / данные

импортируем библиотеки numpy и pandas

In [1]:
import numpy as np
import pandas as pd

задаем некоторые настройки pandas, регулирующие формат вывода

In [2]:
pd.options.display.max_rows = 10

загружаем данные

In [3]:
tips = pd.read_csv("data/tips.csv")
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [4]:
tips['tip_pct'] = tips['tip'] / tips['total_bill']

In [5]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


### Механизм GroupBy

<img src = '../images/split_apply_combine.png' style='width: 600px;'/>

In [6]:
df = pd.DataFrame({'x': ['a','a','b','b','c','c'],
                   'y': [2,4,0,5,5,10]})
df

,x,y
0,a,2
1,a,4
2,b,0
3,b,5
4,c,5
5,c,10


In [7]:
groups = df.groupby(['x'])
groups

получаем информацию о количестве групп, которые будут созданы

In [8]:
groups.ngroups

3

получаем информацию о количестве элементов в каждой группе

In [9]:
groups.size()

x
a    2
b    2
c    2
dtype: int64

что представляют из себя найденные группы?

In [10]:
groups.groups

{'a': [0, 1], 'b': [2, 3], 'c': [4, 5]}

получаем данные конкретной группы

In [11]:
groups.get_group('b')

,x,y
2,b,0
3,b,5


извлекаем первую строку каждой группы

In [12]:
groups.nth([1])

,x,y
1,a,4
3,b,5
5,c,10


обход групп:

In [13]:
for key, group in groups:
    print(key)
    print(group) 

('a',)
   x  y
0  a  2
1  a  4
('b',)
   x  y
2  b  0
3  b  5
('c',)
   x   y
4  c   5
5  c  10


вычисление среднего

In [14]:
groups.y.mean()

x
a    3.0
b    2.5
c    7.5
Name: y, dtype: float64

### Типы группировок

#### по столбцам: 

In [15]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [16]:
tips.groupby(['day','time']).tip.mean().unstack()

time,Dinner,Lunch
day,,
Fri,2.940000,2.382857
Sat,2.993103,NaN
Sun,3.255132,NaN
Thur,3.000000,2.767705


#### по уровням индекса

создаем копию данных и заново индексируем ее

In [17]:
copy_tips = tips.copy()
copy_tips = copy_tips.set_index(['day', 'time'])
copy_tips

total_bill   tip smoker  size   tip_pct
day  time                                           
Sun  Dinner       16.99  1.01     No     2  0.059447
     Dinner       10.34  1.66     No     3  0.160542
     Dinner       21.01  3.50     No     3  0.166587
     Dinner       23.68  3.31     No     2  0.139780
     Dinner       24.59  3.61     No     4  0.146808
...                 ...   ...    ...   ...       ...
Sat  Dinner       29.03  5.92     No     3  0.203927
     Dinner       27.18  2.00    Yes     2  0.073584
     Dinner       22.67  2.00    Yes     2  0.088222
     Dinner       17.82  1.75     No     2  0.098204
Thur Dinner       18.78  3.00     No     2  0.159744

[244 rows x 5 columns]

группировать можем по одному или нескольким уровням индекса, передавая соответствующие значения столбцов аргументу level

In [18]:
copy_tips.groupby(level=['time']).sum()

,total_bill,tip,smoker,size,tip_pct
time,,,,,
Dinner,3660.30,546.07,NoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,463,28.075131
Lunch,1167.47,185.51,NoNoNoYesNoNoYesNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,164,11.160699


In [19]:
copy_tips.groupby(level = ['day', 'time']).mean()

TypeError: agg function failed [how->mean,dtype->object]

#### с использованием функции 

In [20]:
copy_tips.head()

total_bill   tip smoker  size   tip_pct
day time                                           
Sun Dinner       16.99  1.01     No     2  0.059447
    Dinner       10.34  1.66     No     3  0.160542
    Dinner       21.01  3.50     No     3  0.166587
    Dinner       23.68  3.31     No     2  0.139780
    Dinner       24.59  3.61     No     4  0.146808

In [21]:
copy_tips.groupby(len, level='day').count()

,total_bill,tip,smoker,size,tip_pct
day,,,,,
3,182,182,182,182,182
4,62,62,62,62,62


In [22]:
tips.day.value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

#### с использованием массива 

массив случайных меток групп:

In [23]:
np.random.seed(123)
rnd_array = np.random.choice(['first_group', 'second_group'], 
                             size=tips.shape[0],
                             p = [0.4, 0.6])
rnd_array[0:5]

array(['second_group', 'first_group', 'first_group', 'second_group',
       'second_group'], dtype='<U12')

передаем массив меток групп в метод groupby

In [24]:
tips.groupby(rnd_array).count()

,total_bill,tip,smoker,day,time,size,tip_pct
first_group,93,93,93,93,93,93,93
second_group,151,151,151,151,151,151,151


In [25]:
93/(93+151)

0.38114754098360654

#### бонус

тип данных:

In [26]:
tips.dtypes

total_bill    float64
tip           float64
smoker         object
day            object
time           object
size            int64
tip_pct       float64
dtype: object

In [27]:
group_tips = tips.groupby(tips.dtypes, axis=1)

C:\Users\dolgov_vv\AppData\Local\Temp\ipykernel_21676\1239489332.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  group_tips = tips.groupby(tips.dtypes, axis=1)


In [28]:
group_tips.size()

int64      1
float64    3
object     3
dtype: int64

### Агрегирование групп

создаем объект groupby:

In [29]:
group_tips = tips.groupby(['day', 'smoker'])['tip_pct']

проверка эквивалентности методов:

In [30]:
group_tips.agg == group_tips.aggregate

True

собственная функция:

In [31]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

group_tips.agg([peak_to_peak])

peak_to_peak
day  smoker              
Fri  No          0.067349
     Yes         0.159925
Sat  No          0.235193
     Yes         0.290095
Sun  No          0.193226
     Yes         0.644685
Thur No          0.193350
     Yes         0.151240

комбинирование:

In [32]:
group_tips.agg([peak_to_peak, 'mean'])

peak_to_peak      mean
day  smoker                        
Fri  No          0.067349  0.151650
     Yes         0.159925  0.174783
Sat  No          0.235193  0.158048
     Yes         0.290095  0.147906
Sun  No          0.193226  0.160113
     Yes         0.644685  0.187250
Thur No          0.193350  0.160298
     Yes         0.151240  0.163863

задаем имена столбцов:

In [33]:
group_tips.agg([('delta_max_min', peak_to_peak), ('mean_value','mean')])

delta_max_min  mean_value
day  smoker                           
Fri  No           0.067349    0.151650
     Yes          0.159925    0.174783
Sat  No           0.235193    0.158048
     Yes          0.290095    0.147906
Sun  No           0.193226    0.160113
     Yes          0.644685    0.187250
Thur No           0.193350    0.160298
     Yes          0.151240    0.163863

отдельные наборы функций для каждого столбца:

In [34]:
group_tips = tips.groupby(['day', 'smoker'])['tip_pct', 'total_bill']

ValueError: Cannot subset columns with a tuple with more than one element. Use a list instead.

In [ ]:
group_tips.agg({'tip_pct':[('max_value', np.max),
                           ('min_value','min')],
                'total_bill': 'sum'})

### Преобразование групп

#### метод transform

<img src = '../images/transform.png' style='width: 900px;'/>

In [ ]:
df = pd.DataFrame({'Col1': ['A', 'B', 'C', 'C', 'B', 'B', 'A'],
                   'Col2': [1, 2, 3, 4, 2, 5, 3]})
df

In [ ]:
df['Col3'] = df.groupby('Col1').transform(sum)
df.sort_values('Col1')

объект groupby:

In [ ]:
group_tips = tips.groupby(['smoker'])['total_bill']

нормирование:

In [ ]:
norm = lambda x: (x - x.mean())/x.std()
group_tips.transform(norm)

In [ ]:
tips_copy = tips.copy()
tips_copy['total_bill_norm'] = group_tips.transform(norm)
tips_copy.head()

In [ ]:
tips_copy.groupby('smoker').total_bill_norm.agg(['mean', 'std'])

#### метод apply

функция:

In [ ]:
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column)[-n:]
top(tips, n=6)

применяем метод apply

In [ ]:
tips.groupby('smoker').apply(top)

### Исключение групп

создаем данные для наших примеров

In [ ]:
df = pd.DataFrame({'Label': list('AABCCC'),
                   'Values': [1, 2, 3, 4, np.nan, 8]})
df

удаляем группы с менее чем двумя возможными значениеми

In [ ]:
f = lambda x: x.Values.count() > 1
df.groupby('Label').filter(f)

удаляем группы, в которых есть пропуски

In [ ]:
f = lambda x: x.Values.isnull().sum() == 0
df.groupby('Label').filter(f)

### Сводная таблица

средние

In [ ]:
tips.pivot_table(index=['day', 'smoker'])

вывод по строкам и столбцам

In [ ]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker')

включение частичных итогов:

In [ ]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker', margins=True)

задаем функцию:

In [ ]:
tips.pivot_table('tip_pct', index=['time', 'smoker'], columns='day',
                 aggfunc=max, margins=True)